# Estudo global do Radar Financeiro

Este notebook consolida a leitura global do Radar em quatro estudos:

1. Participacao no Radar e motivos de nao participacao.
2. Resultado final, temas vencedores e empates no topo.
3. Orcamento macro/micro e conexao com temas vencedores.
4. Perfil macro/micro e conexao com temas vencedores.

A primeira celula prepara a base analitica reutilizada pelos quatro estudos.

In [ ]:
%%spark

from pyspark.sql import functions as F
from pyspark.sql.window import Window

tabela_origem = globals().get("tabela_spark", "sbx_t2i2016.v1_ana_edu_fin_cli")

df = spark.table(tabela_origem).dropDuplicates(["CD_CLI"])

def texto_normalizado(coluna, padrao="A CLASSIFICAR"):
    return F.upper(F.trim(F.coalesce(F.col(coluna).cast("string"), F.lit(padrao))))

def pct_sobre(coluna_qt, denominador):
    if denominador and denominador > 0:
        return F.round(F.col(coluna_qt) / F.lit(denominador) * 100, 2)

    return F.lit(0.0)

def contar_clientes(df_in, agrupadores, denominador, nome_percentual, ordenar_por=None):
    resultado = (
        df_in
        .groupBy(*agrupadores)
        .agg(F.countDistinct("CD_CLI").alias("Clientes"))
        .withColumn(nome_percentual, pct_sobre("Clientes", denominador))
    )

    if ordenar_por:
        resultado = resultado.orderBy(*ordenar_por)

    return resultado

def mostrar(df_saida, n=200):
    df_saida.show(n, truncate=False)

temas_pontuacao = [
    ("Categorizacao de gastos", "NR_PONT_CATEG", 10),
    ("Gestao do Orcamento", "NR_PONT_ORC", 20),
    ("Consumo Planejado", "NR_PONT_CONS", 30),
    ("Formacao de Reserva", "NR_PONT_RES", 40),
    ("Uso Consciente do Credito", "NR_PONT_CRED", 50),
]

movimentacao_minima = (
    (F.coalesce(F.col("QT_TRANS_TOTAL"), F.lit(0)) > 0)
    & (F.coalesce(F.col("QT_TRANS_ENT"), F.lit(0)) > 0)
    & (F.coalesce(F.col("QT_TRANS_SAI"), F.lit(0)) > 0)
    & (F.coalesce(F.col("VL_TRANS_ENT"), F.lit(0)) > 0)
    & (F.coalesce(F.col("VL_TRANS_SAI"), F.lit(0)) > 0)
)

perfil_renda_valido = ~texto_normalizado("NM_PRFL").isin("A CLASSIFICAR", "SEM PERFIL")
macroperfil_valido = texto_normalizado("NM_MAC_PRFL_CLI") != "A CLASSIFICAR"
microperfil_valido = texto_normalizado("NM_MIC_PRFL_CLI") != "A CLASSIFICAR"
perfil_financeiro_valido = texto_normalizado("NM_PRFL_FIN") != "A CLASSIFICAR"

perfil_completo = (
    perfil_renda_valido
    & macroperfil_valido
    & microperfil_valido
    & perfil_financeiro_valido
)

sem_agro = texto_normalizado("FL_TEM_MOV_AGRO", "N") == "N"
participa_radar = movimentacao_minima & perfil_completo & sem_agro

df_base_estudo = (
    df
    .withColumn("FL_MOV_MINIMA", F.when(movimentacao_minima, "S").otherwise("N"))
    .withColumn("FL_PERFIL_RENDA_VALIDO", F.when(perfil_renda_valido, "S").otherwise("N"))
    .withColumn("FL_MACROPERFIL_VALIDO", F.when(macroperfil_valido, "S").otherwise("N"))
    .withColumn("FL_MICROPERFIL_VALIDO", F.when(microperfil_valido, "S").otherwise("N"))
    .withColumn("FL_PERFIL_FIN_VALIDO", F.when(perfil_financeiro_valido, "S").otherwise("N"))
    .withColumn("FL_PERFIL_COMPLETO", F.when(perfil_completo, "S").otherwise("N"))
    .withColumn("FL_SEM_AGRO", F.when(sem_agro, "S").otherwise("N"))
    .withColumn("FL_PARTICIPA_RADAR_ESTUDO", F.when(participa_radar, "S").otherwise("N"))
    .withColumn(
        "TX_PARTICIPACAO_RADAR",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S", "Participa")
         .otherwise("Nao participa")
    )
    .withColumn("TX_PRFL_ESTUDO", F.coalesce(F.col("NM_PRFL").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_MACROPERFIL_ESTUDO", F.coalesce(F.col("NM_MAC_PRFL_CLI").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_MICROPERFIL_ESTUDO", F.coalesce(F.col("NM_MIC_PRFL_CLI").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_PERFIL_FIN_ESTUDO", F.coalesce(F.col("NM_PRFL_FIN").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_RES_ORC_ESTUDO", F.coalesce(F.col("TX_RES_ORC").cast("string"), F.lit("Nao informado")))
    .withColumn("TX_STS_FINAL_ESTUDO", F.coalesce(F.col("TX_STS_FINAL").cast("string"), F.lit("Nao informado")))
    .withColumn(
        "ORDEM_RES_ORC",
        F.when(F.col("CD_RES_ORC") == 1, 10)
         .when(F.col("CD_RES_ORC") == 0, 20)
         .when(F.col("CD_RES_ORC") == 2, 30)
         .otherwise(99)
    )
    .withColumn(
        "ORDEM_STS_FINAL",
        F.when((F.col("CD_RES_ORC") == 1) & (texto_normalizado("TX_STS_RES", "") == "FORTE"), 10)
         .when((F.col("CD_RES_ORC") == 1) & (texto_normalizado("TX_STS_RES", "") == "FRACO"), 20)
         .when((F.col("CD_RES_ORC") == 0) & (texto_normalizado("TX_STS_RES", "") == "FRACO"), 30)
         .when((F.col("CD_RES_ORC") == 0) & (texto_normalizado("TX_STS_RES", "") == "FORTE"), 40)
         .when((F.col("CD_RES_ORC") == 2) & (texto_normalizado("TX_STS_RES", "") == "FRACO"), 50)
         .when((F.col("CD_RES_ORC") == 2) & (texto_normalizado("TX_STS_RES", "") == "FORTE"), 60)
         .otherwise(99)
    )
)

df_base_estudo = (
    df_base_estudo
    .withColumn(
        "FL_MOTIVO_MOVIMENTACAO",
        F.when(
            (F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N")
            & (F.col("FL_MOV_MINIMA") == "N"),
            "S"
        ).otherwise("N")
    )
    .withColumn(
        "FL_MOTIVO_PERFIL",
        F.when(
            (F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N")
            & (F.col("FL_PERFIL_COMPLETO") == "N"),
            "S"
        ).otherwise("N")
    )
    .withColumn(
        "FL_MOTIVO_AGRO",
        F.when(
            (F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N")
            & (F.col("FL_SEM_AGRO") == "N"),
            "S"
        ).otherwise("N")
    )
    .withColumn(
        "QT_MOTIVOS_NAO_PARTICIPA",
        F.when(
            F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N",
            F.when(F.col("FL_MOTIVO_MOVIMENTACAO") == "S", 1).otherwise(0)
            + F.when(F.col("FL_MOTIVO_PERFIL") == "S", 1).otherwise(0)
            + F.when(F.col("FL_MOTIVO_AGRO") == "S", 1).otherwise(0)
        ).otherwise(0)
    )
    .withColumn(
        "TX_MOTIVO_NAO_PARTICIPA",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S", "Participa")
         .when(F.col("QT_MOTIVOS_NAO_PARTICIPA") > 1, "Multiplos motivos")
         .when(F.col("FL_MOTIVO_MOVIMENTACAO") == "S", "Movimentacao insuficiente")
         .when(F.col("FL_MOTIVO_PERFIL") == "S", "Perfil incompleto")
         .when(F.col("FL_MOTIVO_AGRO") == "S", "Movimentacao Agro")
         .otherwise("Nao classificado")
    )
    .withColumn(
        "TX_COMBINACAO_MOTIVOS",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S", "Participa")
         .otherwise(
             F.concat_ws(
                 " + ",
                 F.when(F.col("FL_MOTIVO_MOVIMENTACAO") == "S", "Movimentacao insuficiente"),
                 F.when(F.col("FL_MOTIVO_PERFIL") == "S", "Perfil incompleto"),
                 F.when(F.col("FL_MOTIVO_AGRO") == "S", "Movimentacao Agro")
             )
         )
    )
)

df_base_estudo = (
    df_base_estudo
    .withColumn(
        "NR_MAIOR_PONTUACAO",
        F.greatest(*[F.coalesce(F.col(coluna).cast("int"), F.lit(0)) for _, coluna, _ in temas_pontuacao])
    )
    .withColumn(
        "ARR_TEMAS_VENCEDORES_RAW",
        F.array(*[
            F.when(
                (F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S")
                & (F.col("NR_MAIOR_PONTUACAO") > 0)
                & (F.coalesce(F.col(coluna).cast("int"), F.lit(0)) == F.col("NR_MAIOR_PONTUACAO")),
                F.lit(tema)
            )
            for tema, coluna, _ in temas_pontuacao
        ])
    )
    .withColumn("ARR_TEMAS_VENCEDORES", F.expr("filter(ARR_TEMAS_VENCEDORES_RAW, x -> x is not null)"))
    .withColumn("QT_TEMAS_VENCEDORES", F.size(F.col("ARR_TEMAS_VENCEDORES")))
    .withColumn(
        "TX_TEMAS_VENCEDORES",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N", "Nao participa")
         .when(F.col("QT_TEMAS_VENCEDORES") == 0, "Sem prioridade")
         .otherwise(F.array_join(F.col("ARR_TEMAS_VENCEDORES"), " + "))
    )
    .withColumn(
        "TX_TEMA_VENCEDOR_UNICO",
        F.when(
            F.col("QT_TEMAS_VENCEDORES") == 1,
            F.element_at(F.col("ARR_TEMAS_VENCEDORES"), 1)
        )
    )
    .withColumn(
        "TX_RESULTADO_FINAL_RADAR",
        F.when(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N", "Nao participa")
         .when(F.col("NR_MAIOR_PONTUACAO") == 0, "Sem prioridade")
         .when(F.col("QT_TEMAS_VENCEDORES") == 1, "Tema vencedor unico")
         .otherwise("Empate no topo")
    )
)

qt_base = df_base_estudo.select(F.countDistinct("CD_CLI").alias("QT_BASE")).first()["QT_BASE"]
qt_participantes = (
    df_base_estudo
    .where(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S")
    .select(F.countDistinct("CD_CLI").alias("QT"))
    .first()["QT"]
)
qt_nao_participantes = qt_base - qt_participantes

linha_periodo = df_base_estudo.select(
    F.min("DT_REF_INI").alias("DT_REF_INI"),
    F.max("DT_REF_FIM").alias("DT_REF_FIM")
).first()

df_participantes = df_base_estudo.where(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "S")
df_vencedor_unico = df_participantes.where(F.col("TX_RESULTADO_FINAL_RADAR") == "Tema vencedor unico")
df_empates = df_participantes.where(F.col("TX_RESULTADO_FINAL_RADAR") == "Empate no topo")

df_base_estudo.createOrReplaceTempView("vw_base_estudo_radar")

print(f"Tabela origem: {tabela_origem}")
print(f"Base analisada: {qt_base} clientes unicos")
print(f"Participantes calculados pelo estudo: {qt_participantes}")
print(f"Nao participantes calculados pelo estudo: {qt_nao_participantes}")
print(f"Periodo: {linha_periodo['DT_REF_INI']} a {linha_periodo['DT_REF_FIM']}")


## 1. Participacao no Radar

Estudo completo de quem participa, quem nao participa e quais motivos explicam a nao participacao.

In [ ]:
%%spark

print("1.1 Regra de participacao")
df_regra_participacao = spark.createDataFrame(
    [
        ("Movimentacao minima", "QT_TRANS_TOTAL > 0"),
        ("Movimentacao minima", "QT_TRANS_ENT > 0"),
        ("Movimentacao minima", "QT_TRANS_SAI > 0"),
        ("Movimentacao minima", "VL_TRANS_ENT > 0"),
        ("Movimentacao minima", "VL_TRANS_SAI > 0"),
        ("Perfil completo", "NM_PRFL NOT IN ('A CLASSIFICAR', 'SEM PERFIL')"),
        ("Perfil completo", "NM_MAC_PRFL_CLI <> 'A CLASSIFICAR'"),
        ("Perfil completo", "NM_MIC_PRFL_CLI <> 'A CLASSIFICAR'"),
        ("Perfil completo", "NM_PRFL_FIN <> 'A CLASSIFICAR'"),
        ("Exclusao Agro", "FL_TEM_MOV_AGRO = 'N'"),
    ],
    ["Bloco", "Criterio tecnico"]
)
mostrar(df_regra_participacao)

print("1.2 Resumo de participacao")
df_resumo_participacao = (
    contar_clientes(
        df_base_estudo,
        ["TX_PARTICIPACAO_RADAR"],
        qt_base,
        "% Base"
    )
    .withColumn(
        "ORDEM",
        F.when(F.col("TX_PARTICIPACAO_RADAR") == "Participa", 10).otherwise(20)
    )
    .orderBy("ORDEM")
    .select(
        F.col("TX_PARTICIPACAO_RADAR").alias("Situacao"),
        "Clientes",
        "% Base"
    )
)
mostrar(df_resumo_participacao)

print("1.3 Blocos de elegibilidade")
linhas_elegibilidade = [
    F.struct(
        F.lit(10).alias("ORDEM"),
        F.lit("Movimentacao minima").alias("Bloco"),
        F.when(F.col("FL_MOV_MINIMA") == "S", "Atende").otherwise("Nao atende").alias("Situacao")
    ),
    F.struct(
        F.lit(20).alias("ORDEM"),
        F.lit("Perfil completo").alias("Bloco"),
        F.when(F.col("FL_PERFIL_COMPLETO") == "S", "Atende").otherwise("Nao atende").alias("Situacao")
    ),
    F.struct(
        F.lit(30).alias("ORDEM"),
        F.lit("Movimentacao Agro").alias("Bloco"),
        F.when(F.col("FL_SEM_AGRO") == "S", "Sem Agro").otherwise("Com Agro").alias("Situacao")
    ),
]

df_blocos_elegibilidade = (
    df_base_estudo
    .select("CD_CLI", F.explode(F.array(*linhas_elegibilidade)).alias("B"))
    .select(
        "CD_CLI",
        F.col("B.ORDEM").alias("ORDEM"),
        F.col("B.Bloco").alias("Bloco"),
        F.col("B.Situacao").alias("Situacao")
    )
)

df_blocos_elegibilidade_exibicao = (
    contar_clientes(df_blocos_elegibilidade, ["ORDEM", "Bloco", "Situacao"], qt_base, "% Base", ["ORDEM", "Situacao"])
    .select("Bloco", "Situacao", "Clientes", "% Base")
)
mostrar(df_blocos_elegibilidade_exibicao)

print("1.4 Motivos agrupados de nao participacao")
df_motivos_agrupados = (
    contar_clientes(
        df_base_estudo.where(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N"),
        ["TX_MOTIVO_NAO_PARTICIPA"],
        qt_base,
        "% Base"
    )
    .withColumn("% Nao participantes", pct_sobre("Clientes", qt_nao_participantes))
    .withColumn(
        "Regra resumida",
        F.when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Movimentacao insuficiente", "Falhou em QT/VL de entrada ou saida")
         .when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Perfil incompleto", "Algum perfil esta A CLASSIFICAR ou NM_PRFL esta SEM PERFIL")
         .when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Movimentacao Agro", "FL_TEM_MOV_AGRO = 'S'")
         .when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Multiplos motivos", "Falhou em mais de um bloco")
         .otherwise("Nao classificado")
    )
    .withColumn(
        "ORDEM",
        F.when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Movimentacao insuficiente", 10)
         .when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Perfil incompleto", 20)
         .when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Movimentacao Agro", 30)
         .when(F.col("TX_MOTIVO_NAO_PARTICIPA") == "Multiplos motivos", 40)
         .otherwise(99)
    )
    .orderBy("ORDEM")
    .select(
        F.col("TX_MOTIVO_NAO_PARTICIPA").alias("Motivo"),
        "Regra resumida",
        "Clientes",
        "% Base",
        "% Nao participantes"
    )
)
mostrar(df_motivos_agrupados)

print("1.5 Todos os motivos identificados, com sobreposicao")
linhas_motivos = [
    F.struct(
        F.lit(10).alias("ORDEM"),
        F.lit("Movimentacao insuficiente").alias("Motivo"),
        F.lit("Falhou em QT/VL de entrada ou saida").alias("Regra resumida"),
        (F.col("FL_MOTIVO_MOVIMENTACAO") == "S").alias("FL_MOTIVO")
    ),
    F.struct(
        F.lit(20).alias("ORDEM"),
        F.lit("Perfil incompleto").alias("Motivo"),
        F.lit("Algum perfil esta A CLASSIFICAR ou NM_PRFL esta SEM PERFIL").alias("Regra resumida"),
        (F.col("FL_MOTIVO_PERFIL") == "S").alias("FL_MOTIVO")
    ),
    F.struct(
        F.lit(30).alias("ORDEM"),
        F.lit("Movimentacao Agro").alias("Motivo"),
        F.lit("FL_TEM_MOV_AGRO = 'S'").alias("Regra resumida"),
        (F.col("FL_MOTIVO_AGRO") == "S").alias("FL_MOTIVO")
    ),
]

df_motivos_individuais = (
    df_base_estudo
    .select("CD_CLI", F.explode(F.array(*linhas_motivos)).alias("M"))
    .where(F.col("M.FL_MOTIVO"))
    .select(
        "CD_CLI",
        F.col("M.ORDEM").alias("ORDEM"),
        F.col("M.Motivo").alias("Motivo"),
        F.col("M.Regra resumida").alias("Regra resumida")
    )
)

df_motivos_individuais_exibicao = (
    contar_clientes(df_motivos_individuais, ["ORDEM", "Motivo", "Regra resumida"], qt_base, "% Base", ["ORDEM"])
    .withColumn("% Nao participantes", pct_sobre("Clientes", qt_nao_participantes))
    .select("Motivo", "Regra resumida", "Clientes", "% Base", "% Nao participantes")
)
mostrar(df_motivos_individuais_exibicao)

print("1.6 Combinacao dos motivos de nao participacao")
df_combinacao_motivos = (
    contar_clientes(
        df_base_estudo.where(F.col("FL_PARTICIPA_RADAR_ESTUDO") == "N"),
        ["TX_COMBINACAO_MOTIVOS"],
        qt_nao_participantes,
        "% Nao participantes",
        [F.desc("Clientes")]
    )
    .select(F.col("TX_COMBINACAO_MOTIVOS").alias("Combinacao de motivos"), "Clientes", "% Nao participantes")
)
mostrar(df_combinacao_motivos)


## 2. Resultado final e temas vencedores

Estudo global da decisao final do Radar, calculando tema vencedor unico, empates no topo e quantidade de temas empatados.

In [ ]:
%%spark

print("2.1 Regra de tema vencedor")
df_regra_temas = spark.createDataFrame(
    [(tema, coluna) for tema, coluna, _ in temas_pontuacao],
    ["Tema", "Coluna de pontuacao final"]
)
mostrar(df_regra_temas)
print("Classificacao: vencedor unico = um tema isolado com a maior pontuacao; empate = dois ou mais temas no topo; sem prioridade = maior pontuacao igual a zero.")

qt_vencedor_unico = df_vencedor_unico.select(F.countDistinct("CD_CLI").alias("QT")).first()["QT"]
qt_empates = df_empates.select(F.countDistinct("CD_CLI").alias("QT")).first()["QT"]

print("2.2 Resumo da decisao final")
df_resumo_decisao = (
    contar_clientes(
        df_participantes,
        ["TX_RESULTADO_FINAL_RADAR"],
        qt_participantes,
        "% Participantes"
    )
    .withColumn(
        "ORDEM",
        F.when(F.col("TX_RESULTADO_FINAL_RADAR") == "Tema vencedor unico", 10)
         .when(F.col("TX_RESULTADO_FINAL_RADAR") == "Empate no topo", 20)
         .when(F.col("TX_RESULTADO_FINAL_RADAR") == "Sem prioridade", 30)
         .otherwise(99)
    )
    .orderBy("ORDEM")
    .select(F.col("TX_RESULTADO_FINAL_RADAR").alias("Resultado final"), "Clientes", "% Participantes")
)
mostrar(df_resumo_decisao)

print("2.3 Quantidade de temas vencedores no topo")
df_qtd_temas_topo = (
    contar_clientes(
        df_participantes,
        ["QT_TEMAS_VENCEDORES"],
        qt_participantes,
        "% Participantes",
        ["QT_TEMAS_VENCEDORES"]
    )
    .withColumn(
        "Qtd temas no topo",
        F.concat(
            F.col("QT_TEMAS_VENCEDORES").cast("string"),
            F.when(F.col("QT_TEMAS_VENCEDORES") == 1, F.lit(" tema")).otherwise(F.lit(" temas"))
        )
    )
    .select("Qtd temas no topo", "Clientes", "% Participantes")
)
mostrar(df_qtd_temas_topo)

print("2.4 Temas vencedores unicos")
df_temas_vencedores_unicos = (
    contar_clientes(
        df_vencedor_unico,
        ["TX_TEMA_VENCEDOR_UNICO"],
        qt_participantes,
        "% Participantes",
        [F.desc("Clientes")]
    )
    .withColumn("% Vencedor unico", pct_sobre("Clientes", qt_vencedor_unico))
    .select(F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema vencedor"), "Clientes", "% Participantes", "% Vencedor unico")
)
mostrar(df_temas_vencedores_unicos)

print("2.5 Temas no topo, incluindo clientes empatados")
df_temas_no_topo = (
    df_participantes
    .where(F.col("QT_TEMAS_VENCEDORES") > 0)
    .select("CD_CLI", F.explode("ARR_TEMAS_VENCEDORES").alias("Tema no topo"))
)

df_temas_no_topo_exibicao = (
    contar_clientes(df_temas_no_topo, ["Tema no topo"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select("Tema no topo", "Clientes", "% Participantes")
)
mostrar(df_temas_no_topo_exibicao)
print("Obs.: em caso de empate, o cliente conta em todos os temas empatados nesta tabela.")

print("2.6 Top 3 empates no topo")
df_top3_empates = (
    contar_clientes(
        df_empates,
        ["TX_TEMAS_VENCEDORES"],
        qt_participantes,
        "% Participantes",
        [F.desc("Clientes")]
    )
    .withColumn("% Empates", pct_sobre("Clientes", qt_empates))
    .select(F.col("TX_TEMAS_VENCEDORES").alias("Temas empatados no topo"), "Clientes", "% Participantes", "% Empates")
    .limit(3)
)
mostrar(df_top3_empates)


## 3. Orcamento x temas vencedores

Estudo do resultado orcamentario em modo macro e micro, conectado aos temas vencedores unicos.

In [ ]:
%%spark

print("3.1 Regra do resultado orcamentario")
df_regra_orcamento = spark.createDataFrame(
    [
        ("Indicador base", "PC_SAI_ENT = VL_SAI_TOTAL / VL_ENT_TOTAL"),
        ("< 0,75", "Superavitario Forte"),
        ("0,75 ate < 0,95", "Superavitario Fraco"),
        ("0,95 ate < 1,00", "Neutro Fraco"),
        ("1,00 ate 1,05", "Neutro Forte"),
        ("> 1,05 ate 1,25", "Deficitario Fraco"),
        ("> 1,25", "Deficitario Forte"),
    ],
    ["Faixa / indicador", "Resultado"]
)
mostrar(df_regra_orcamento)

print("3.2 Orcamento macro")
df_orcamento_macro = (
    df_participantes
    .groupBy("TX_RES_ORC_ESTUDO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_RES_ORC").alias("ORDEM")
    )
    .withColumn("% Participantes", pct_sobre("Clientes", qt_participantes))
    .orderBy("ORDEM")
    .select(F.col("TX_RES_ORC_ESTUDO").alias("Resultado orcamentario"), "Clientes", "% Participantes")
)
mostrar(df_orcamento_macro)

print("3.3 Orcamento micro")
df_orcamento_micro = (
    df_participantes
    .groupBy("TX_STS_FINAL_ESTUDO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_STS_FINAL").alias("ORDEM")
    )
    .withColumn("% Participantes", pct_sobre("Clientes", qt_participantes))
    .orderBy("ORDEM")
    .select(F.col("TX_STS_FINAL_ESTUDO").alias("Status orcamentario"), "Clientes", "% Participantes")
)
mostrar(df_orcamento_micro)

print("3.4 Orcamento macro x tema vencedor unico")
df_orc_macro_tema = (
    df_vencedor_unico
    .groupBy("TX_RES_ORC_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_RES_ORC").alias("ORDEM")
    )
    .withColumn(
        "% do resultado",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_RES_ORC_ESTUDO")) * 100, 2)
    )
    .orderBy("ORDEM", F.desc("Clientes"))
    .select(
        F.col("TX_RES_ORC_ESTUDO").alias("Resultado orcamentario"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema vencedor"),
        "Clientes",
        "% do resultado"
    )
)
mostrar(df_orc_macro_tema)

print("3.5 Orcamento micro x tema vencedor unico")
df_orc_micro_tema = (
    df_vencedor_unico
    .groupBy("TX_STS_FINAL_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(
        F.countDistinct("CD_CLI").alias("Clientes"),
        F.min("ORDEM_STS_FINAL").alias("ORDEM")
    )
    .withColumn(
        "% do status",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_STS_FINAL_ESTUDO")) * 100, 2)
    )
    .orderBy("ORDEM", F.desc("Clientes"))
    .select(
        F.col("TX_STS_FINAL_ESTUDO").alias("Status orcamentario"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema vencedor"),
        "Clientes",
        "% do status"
    )
)
mostrar(df_orc_micro_tema)
print("Obs.: as tabelas de conexao com tema vencedor consideram apenas clientes com tema vencedor unico.")


## 4. Perfil x temas vencedores

Estudo de perfil de renda, macroperfil, microperfil e perfil financeiro, conectado aos temas vencedores unicos.

In [ ]:
%%spark

print("4.1 Regra de perfil completo")
df_regra_perfil = spark.createDataFrame(
    [
        ("NM_PRFL", "NOT IN ('A CLASSIFICAR', 'SEM PERFIL')"),
        ("NM_MAC_PRFL_CLI", "<> 'A CLASSIFICAR'"),
        ("NM_MIC_PRFL_CLI", "<> 'A CLASSIFICAR'"),
        ("NM_PRFL_FIN", "<> 'A CLASSIFICAR'"),
    ],
    ["Campo", "Criterio tecnico"]
)
mostrar(df_regra_perfil)

print("4.2 Perfil de renda dos participantes")
df_perfil_renda = (
    contar_clientes(df_participantes, ["TX_PRFL_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_PRFL_ESTUDO").alias("Perfil de renda"), "Clientes", "% Participantes")
)
mostrar(df_perfil_renda)

print("4.3 Macroperfil dos participantes")
df_macroperfil = (
    contar_clientes(df_participantes, ["TX_MACROPERFIL_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_MACROPERFIL_ESTUDO").alias("Macroperfil"), "Clientes", "% Participantes")
)
mostrar(df_macroperfil)

print("4.4 Microperfil dos participantes")
df_microperfil = (
    contar_clientes(df_participantes, ["TX_MICROPERFIL_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_MICROPERFIL_ESTUDO").alias("Microperfil"), "Clientes", "% Participantes")
)
mostrar(df_microperfil)

print("4.5 Perfil financeiro dos participantes")
df_perfil_financeiro = (
    contar_clientes(df_participantes, ["TX_PERFIL_FIN_ESTUDO"], qt_participantes, "% Participantes", [F.desc("Clientes")])
    .select(F.col("TX_PERFIL_FIN_ESTUDO").alias("Perfil financeiro"), "Clientes", "% Participantes")
)
mostrar(df_perfil_financeiro)

print("4.6 Macroperfil x tema vencedor unico")
df_macro_tema = (
    df_vencedor_unico
    .groupBy("TX_MACROPERFIL_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
    .withColumn(
        "% do macroperfil",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_MACROPERFIL_ESTUDO")) * 100, 2)
    )
    .orderBy("TX_MACROPERFIL_ESTUDO", F.desc("Clientes"))
    .select(
        F.col("TX_MACROPERFIL_ESTUDO").alias("Macroperfil"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema vencedor"),
        "Clientes",
        "% do macroperfil"
    )
)
mostrar(df_macro_tema)

print("4.7 Microperfil x tema vencedor unico")
df_micro_tema = (
    df_vencedor_unico
    .groupBy("TX_MICROPERFIL_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
    .withColumn(
        "% do microperfil",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_MICROPERFIL_ESTUDO")) * 100, 2)
    )
    .orderBy("TX_MICROPERFIL_ESTUDO", F.desc("Clientes"))
    .select(
        F.col("TX_MICROPERFIL_ESTUDO").alias("Microperfil"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema vencedor"),
        "Clientes",
        "% do microperfil"
    )
)
mostrar(df_micro_tema)

print("4.8 Perfil financeiro x tema vencedor unico")
df_perfil_fin_tema = (
    df_vencedor_unico
    .groupBy("TX_PERFIL_FIN_ESTUDO", "TX_TEMA_VENCEDOR_UNICO")
    .agg(F.countDistinct("CD_CLI").alias("Clientes"))
    .withColumn(
        "% do perfil",
        F.round(F.col("Clientes") / F.sum("Clientes").over(Window.partitionBy("TX_PERFIL_FIN_ESTUDO")) * 100, 2)
    )
    .orderBy("TX_PERFIL_FIN_ESTUDO", F.desc("Clientes"))
    .select(
        F.col("TX_PERFIL_FIN_ESTUDO").alias("Perfil financeiro"),
        F.col("TX_TEMA_VENCEDOR_UNICO").alias("Tema vencedor"),
        "Clientes",
        "% do perfil"
    )
)
mostrar(df_perfil_fin_tema)
print("Obs.: as tabelas de conexao com tema vencedor consideram apenas clientes com tema vencedor unico.")
